# Projet : Stage
### Objectif

Pour les attributs spatiaux et temporels, on voudrait faire la même chose. Pour cette partie, il y a 3 étapes à faire : 

1. Identifier les attributs spatiaux / temporels. 
2. Trouver le niveau d'hiérarchie de chaque attributs selon les hiérarchies spatiales / temporelles
3. Identifier le niveau d'hiérarchie le plus fin parmis tous les attributs spatiaux / temporels en tant que granularité minimum de dataset; identifier l'attribut au niveau d'hiérarchie le plus haut parmis tous les attributs en tant que scope de dataset et donner la liste de ses valeurs distinctes. 

L'output final qu'on demande est un dossier json de métadonnée de tous les datasets.

## 1. Hiérarchisation des données

In [1]:
import os
import re
import json
import csv
import pandas as pd
import load_file as lf

def construire_dictionnaire_hierarchise():

    def fillChamp(dicChamps, dic_hierarchise, rang):
        for i in range(len(dicChamps['features'])):
            champ = dicChamps['features'][i]['properties']['nom']
            if champ not in dic_hierarchise[rang]:
                dic_hierarchise[rang].append(champ.lower())
        return

    def fillDictionnaireGeoJSON():
        fichiers = ['communes', 'departements', 'regions']
        dic_hierarchise = {}

        for fichier in fichiers:
            dic_hierarchise[fichier] = []
            with open(f"levels/france-geojson/{fichier}-avec-outre-mer.geojson", "r", encoding="utf-8") as mon_json:
                data = json.load(mon_json)
                fillChamp(data, dic_hierarchise, fichier)

        return dic_hierarchise

    def fillDictionnaireQuartiers(dic_hierarchise):
        with open("liste-correspondance-qp2024-qp2015.csv", "r", encoding="utf-8") as fichier:
            reader = csv.reader(fichier, delimiter=";")
            listeQuartiers = list(reader)[1:]  # Ignorer l'en-tête

        dic_hierarchise['quartiers'] = []
        for i in range(len(listeQuartiers)):
            quartier = listeQuartiers[i][1]
            if quartier not in dic_hierarchise['quartiers'] and quartier != "":
                dic_hierarchise['quartiers'].append(quartier.lower())

        dic_hierarchise['QP'] = []
        for i in range(len(listeQuartiers)):
            quartier = listeQuartiers[i][3]
            if quartier not in dic_hierarchise['QP'] and quartier != "":
                dic_hierarchise['quartiers'].append(quartier.lower())

    dic_hierarchise = fillDictionnaireGeoJSON()
    fillDictionnaireQuartiers(dic_hierarchise)    

    # Tri des listes dans le dictionnaire
    champs = ['regions', 'departements', 'communes', 'cantons', 'quartiers', 'QP']
    dic_hierarchise = {champ: dic_hierarchise[champ] for champ in champs if champ in dic_hierarchise}
    dic_hierarchise['pays'] = ['france', 'france métropolitaine', 'france d\'outre-mer', 'france entière']
    
    return dic_hierarchise

def recuperer_dictionnaire_hierarchise():
    try:
        with open("dic_hierarchise.json", "r", encoding="utf-8") as fichier:
            dic_hierarchise = json.load(fichier)
    except FileNotFoundError:
        dic_hierarchise = construire_dictionnaire_hierarchise()
        with open("dic_hierarchise.json", "w", encoding="utf-8") as fichier:
            json.dump(dic_hierarchise, fichier, ensure_ascii=False, indent=4)
    
    return dic_hierarchise

In [2]:
# Appel de la fonction pour obtenir le dictionnaire hiérarchisé
dic_hierarchise = recuperer_dictionnaire_hierarchise()
# champs_ranges = ['regions', 'departements', 'communes', 'cantons', 'quartiers', 'QP', 'geopoint']
champs_ranges = ['pays', 'regions', 'departements', 'quartiers', 'communes', 'iris', 'geopoints']
temps_ranges = ['annee', 'trimestre', 'mois', 'semaine', 'date']
hierarchie_champs_spa = {champ: (len(champs_ranges) - i) for i, champ in enumerate(champs_ranges)}
hierarchie_temps_spa = {temps: (len(temps_ranges) - i) for i, temps in enumerate(temps_ranges)}

## 2. Identification des attributs spatiaux dans un fichier csv/xlsx

### 2.1 Récupérer tous les attributs spatiaux

#### 2.1.1 Recuperation de tous les datasets

In [3]:
import os

def getFiles(origine='Opendata'):
    fichiers = []

    def separateurFichier(datasets):
        datasetSepares = {}

        for dataset in datasets:
            if dataset.endswith('.csv'):
                if 'csv' not in datasetSepares:
                    datasetSepares['csv'] = []
                datasetSepares['csv'].append(dataset)

            elif dataset.endswith('.xlsx'):
                if 'xlsx' not in datasetSepares:
                    datasetSepares['xlsx'] = []
                datasetSepares['xlsx'].append(dataset)

        return datasetSepares

    for dossier in os.walk(origine):
        for fichier in dossier[2]:
            if fichier.endswith('.csv') or fichier.endswith('.xlsx'):
                fichiers.append(os.path.join(dossier[0], fichier))
    
    return fichiers

datasets = getFiles()

#### 2.1.2 Recherche des attributs spatiaux via contenu des cellules

Variables utiles

In [4]:
regexAnnee = r'(19\d{2}|20\d{2})$'
regexMois = r'(0[1-9]|1[0-2])'
regexJour = r'(0[1-9]|[12]\d|3[01])'
regexDate = r'(' + regexAnnee[:-1] + r'[-/]?' + regexMois + r'[-/]?' + regexJour + r')'
regexHeure = r'([01]\d|2[0-3]):([0-5]\d):([0-5]\d)'
regexTrim = r'(19\d{2}|20\d{2})_[a-zA-Z]{1}[1-3]'
regexGeopoint1 = r'^(-?\d+(?:\.\d+)?)[,; ]\s*(-?\d+(?:\.\d+)?)$'
regexGeopoint2 = r'^(-?\d+(?:\.\d+)?)\s+(-?\d+(?:\.\d+)?)$'

listeRegexTemporel = [[regexDate, 'date'], [regexHeure, 'heure'], [regexAnnee, 'annee'], [regexTrim, 'trimestre']]
listeRegexGeopoint = [[regexGeopoint1, 'geopoints'], [regexGeopoint2, 'geopoints']]

iris_df = pd.read_csv('table_passage_1999_2022.csv', sep=',', encoding='utf-8')
iris_values = set(iris_df.values.flatten().astype(str))

Fonctions utiles

In [5]:
def estGeopoint(cell):
    cell = str(cell).strip()
    match = re.match(r'^(-?\d+(?:\.\d+)?)[,; ]\s*(-?\d+(?:\.\d+)?)$', cell)
    if match:
        lat, lon = float(match.group(1)), float(match.group(2))
        if -90 <= lat <= 90 and -180 <= lon <= 180:
            return [True, 'geopoints']
    return [False, None]

def estIRIS(cell):
    return [True, 'iris'] if str(cell) in iris_values and str(cell) != 'nan' else [False, None]

def estSpatial(cell):
    cell = str(cell)
    infoIRIS = estIRIS(cell)
    if infoIRIS[0]:
        return infoIRIS
    infoGeopoint = estGeopoint(cell)
    if infoGeopoint[0]:
        return infoGeopoint
    for champ, valeurs in dic_hierarchise.items():
        if cell in valeurs:
            return [True, champ]
        if '-' in cell:
            allCell = cell.split('-')
            for cell_n in allCell:
                if cell_n in valeurs:
                    return [True, champ]
    return [False, None]

def estTemporel(cell):
    if not isinstance(cell, str):
        cell = str(cell)
    for regex, label in listeRegexTemporel:
        if re.match(regex, cell):
            return [True, label]
    if cell.lower() in ['janvier', 'fevrier', 'mars', 'avril', 'mai', 'juin', 'juillet', 'aout', 'septembre', 'octobre', 'novembre', 'decembre']:
        return [True, 'mois']
    return [False, None]

def recupererAttributsSpatiaux(headers, df, score_colonne):
    liste_attributs_spatiaux = {}
    n_rows = min(10, len(df))
    for j, header in enumerate(headers):
        found = False
        for i in range(n_rows):
            cell = df.iloc[i, j]
            if isinstance(cell, str):
                cell = cell.lower()
            infoSpatial = estSpatial(cell)
            if infoSpatial[0]:
                score_colonne[j] += 1
                if (score_colonne[j]*10) >= 50 and header not in liste_attributs_spatiaux:
                    liste_attributs_spatiaux[header] = [cell, infoSpatial[1]]
                    found = True
                    break
        if found:
            continue
    return liste_attributs_spatiaux

def recupererAttributsTemporels(headers, df, score_colonne):
    liste_attributs_temporels = {}
    n_rows = min(10, len(df))
    for j, header in enumerate(headers):
        found = False
        for i in range(n_rows):
            cell = df.iloc[i, j]
            if not isinstance(cell, str):
                cell = str(cell)
            infoTemporel = estTemporel(cell)
            if infoTemporel[0]:
                score_colonne[j] += 1
                if (score_colonne[j]*10) >= 50 and header not in liste_attributs_temporels:
                    liste_attributs_temporels[header] = [cell, infoTemporel[1]]
                    found = True
                    break  
        if found:
            continue
    return liste_attributs_temporels

def rechercherLowGranEtScope(liste_attributs, hierarchie={}):
    max_att = list(hierarchie.keys())[-1]
    min_att = list(hierarchie.keys())[0]

    le_plus_bas = [min_att, None]
    le_plus_haut = [max_att, None]

    for champ, valeur in liste_attributs.items():
        if hierarchie[valeur[1]] <= hierarchie[le_plus_bas[0]]:
            le_plus_bas = [valeur[1], champ]
        if hierarchie[valeur[1]] >= hierarchie[le_plus_haut[0]]:
            le_plus_haut = [valeur[1], champ]

    return {'LowGranularite': le_plus_bas, 'Scope': le_plus_haut}

def chercherEntete(df, max_lignes=20):
    lignes_testees = 0
    old_df = None
    while lignes_testees < max_lignes:
        headers = df.columns.tolist()
        headers_valides = True
        for h in headers:
            if re.match(r'.*(U|u)nnamed.*', str(h)):
                headers_valides = False
            if ' ' in str(h).strip():
                headers_valides = False
        if headers_valides:
            for header in headers:
                if (estTemporel(header)[0] or estSpatial(header)[0]) and old_df is not None:
                    df = old_df.copy()
                    break
            return df, False
        if len(df) < 1:
            break
        new_headers = df.iloc[0].tolist()
        old_df = df.copy()
        df = df[1:].copy()
        df.columns = [str(h) for h in new_headers]
        lignes_testees += 1
    return df, True


def process_dataframe(df, nom_fichier, hierarchie_temps_spa, hierarchie_champs_spa):
    headers = df.columns.tolist()
    score_colonne = {k: 0 for k in range(len(headers))}
    df_sample = df.head(10).copy()
    df_sample = df_sample.astype('object')
    for col in df_sample.columns:
        df_sample.loc[:, col] = df_sample[col].map(lambda x: str(x).lower() if isinstance(x, str) else str(x))

    liste_attributs_spatiaux = recupererAttributsSpatiaux(headers, df_sample, score_colonne)
    liste_attributs_temporels = recupererAttributsTemporels(headers, df_sample, score_colonne)

    low_gran_and_scope_tem = {nom_fichier: rechercherLowGranEtScope(liste_attributs_temporels, hierarchie_temps_spa)}
    low_gran_and_scope_spa = {nom_fichier: rechercherLowGranEtScope(liste_attributs_spatiaux, hierarchie_champs_spa)}

    print(f"Attributs temporels : {liste_attributs_temporels}")
    print(f"Attributs spatiaux : {liste_attributs_spatiaux}")
    print(f"LowGranularite et Scope temporels : {low_gran_and_scope_tem[nom_fichier]}")
    print(f"LowGranularite et Scope spatiaux : {low_gran_and_scope_spa[nom_fichier]}\n")
    return low_gran_and_scope_tem, low_gran_and_scope_spa

In [ ]:
compteur_datasets = 0

valeurs_spatiales = set()
for valeurs in dic_hierarchise.values():
    valeurs_spatiales.update(valeurs)

for dataset in datasets:
    nom_fichier = dataset.split('/')[-1]
    extension = nom_fichier.split('.')[-1]
    compteur_datasets += 1
    print(f"Dataset {compteur_datasets}/ {len(datasets)} : {nom_fichier}")

    try:
        if extension == 'xlsx':
            listeSheets = pd.ExcelFile(dataset)
            nbSheets = len(listeSheets.sheet_names)
            print(f'Fichier Excel | {nbSheets}\n')
            for i in range(nbSheets):
                try:
                    df = lf.find_type(dataset, i)[0]
                    df, feuilleInvalides = chercherEntete(df)
                    if feuilleInvalides:
                        print(f"Feuille {i} invalide, passage à la suivante.")
                        continue
                    low_gran_and_scope_tem, low_gran_and_scope_spa = process_dataframe(
                        df, f"{nom_fichier}_sheet{i}", hierarchie_temps_spa, hierarchie_champs_spa
                    )
                except Exception as e:
                    print(f"Erreur lors de la lecture de la feuille {i} : {e}")
                    continue
        elif extension == 'csv':
            print('Fichier CSV\n')
            try:
                df = lf.find_type(dataset)[0]
                if len(df.columns) < 5:
                    try:
                        df = pd.read_csv(dataset, sep=';', encoding='utf-8', nrows=500)
                    except:
                        df = pd.read_csv(dataset, sep=';', encoding='latin1', nrows=500)

                low_gran_and_scope_tem, low_gran_and_scope_spa = process_dataframe(
                    df, nom_fichier, hierarchie_temps_spa, hierarchie_champs_spa
                )
            except Exception as e:
                print(f"Erreur lors du chargement du fichier {nom_fichier[:10]}: {e}")
                continue
        else:
            print(f"Extension non supportée pour {nom_fichier}")
            continue
    except Exception as e:
        print(f"Erreur générale sur {nom_fichier}: {e}")
        continue

Dataset 1/ 72 : fr-en-adresse-et-geolocalisation-etablissements-premier-et-second-degre.csv
Fichier CSV

Attributs temporels : {'date_ouverture': ['1971-05-24', 'date']}
Attributs spatiaux : {'localite_acheminement_uai': ['villemomble', 'communes'], 'libelle_commune': ['tremblay-en-france', 'communes'], 'libelle_departement': ['seine-saint-denis', 'departements'], 'libelle_region': ['ile-de-france', 'regions'], 'libelle_academie': ['créteil', 'communes'], 'position': ['48.95679235130588, 2.583823020574583', 'geopoints']}
LowGranularite et Scope temporels : {'LowGranularite': ['date', 'date_ouverture'], 'Scope': ['date', 'date_ouverture']}
LowGranularite et Scope spatiaux : {'LowGranularite': ['geopoints', 'position'], 'Scope': ['regions', 'libelle_region']}

Dataset 2/ 72 : landes-aggregated-gtfs.xlsx
Fichier Excel | 8

Feuille 0 invalide, passage à la suivante.
Attributs temporels : {20240101: ['20240101', 'date'], 20251231: ['20251231', 'date']}
Attributs spatiaux : {}
LowGranularite

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7a6db88a5af0>>
Traceback (most recent call last):
  File "/home/thibault/Documents/Stage/stage/lib/python3.12/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


Attributs temporels : {}
Attributs spatiaux : {'CODGEO': ['qp001007', 'quartiers']}
LowGranularite et Scope temporels : {'LowGranularite': ['annee', None], 'Scope': ['date', None]}
LowGranularite et Scope spatiaux : {'LowGranularite': ['quartiers', 'CODGEO'], 'Scope': ['quartiers', 'CODGEO']}

Attributs temporels : {}
Attributs spatiaux : {}
LowGranularite et Scope temporels : {'LowGranularite': ['annee', None], 'Scope': ['date', None]}
LowGranularite et Scope spatiaux : {'LowGranularite': ['pays', None], 'Scope': ['geopoints', None]}

Attributs temporels : {}
Attributs spatiaux : {'LIBGEO': ['ferney-voltaire', 'communes']}
LowGranularite et Scope temporels : {'LowGranularite': ['annee', None], 'Scope': ['date', None]}
LowGranularite et Scope spatiaux : {'LowGranularite': ['communes', 'LIBGEO'], 'Scope': ['communes', 'LIBGEO']}

Attributs temporels : {}
Attributs spatiaux : {}
LowGranularite et Scope temporels : {'LowGranularite': ['annee', None], 'Scope': ['date', None]}
LowGranularit

In [ ]:
# import load_file as lf
# fichier = 'Opendata/Général/Santé/Les bénéficiaires de l_aide sociale à l_enfance - séries longues (1996-2022).xlsx'
# nom_fichier = fichier.split('/')[-1]
# score_colonne = {}
# nbSheets = len(pd.ExcelFile(fichier).sheet_names)
# for i in range(nbSheets):
#     try:
#         df = lf.find_type(fichier, i)[0]
#         df, feuilleInvalides = chercherEntete(df)
#         if feuilleInvalides:
#             print(f"Feuille {i} invalide, passage à la suivante.")
#             continue
#         low_gran_and_scope_tem, low_gran_and_scope_spa = process_dataframe(
#             df, f"{nom_fichier}_sheet{i}", hierarchie_temps_spa, hierarchie_champs_spa
#         )
#     except Exception as e:
#         print(f"Erreur lors de la lecture de la feuille {i} : {e}")


Attributs temporels : {}
Attributs spatiaux : {}
LowGranularite et Scope temporels : {'LowGranularite': ['annee', None], 'Scope': ['date', None]}
LowGranularite et Scope spatiaux : {'LowGranularite': ['pays', None], 'Scope': ['geopoints', None]}

Attributs temporels : {}
Attributs spatiaux : {}
LowGranularite et Scope temporels : {'LowGranularite': ['annee', None], 'Scope': ['date', None]}
LowGranularite et Scope spatiaux : {'LowGranularite': ['pays', None], 'Scope': ['geopoints', None]}

Erreur lors de la lecture de la feuille 2 : Setting with non-unique columns is not allowed.
Attributs temporels : {}
Attributs spatiaux : {'Ain': ['alpes-maritimes', 'departements']}
LowGranularite et Scope temporels : {'LowGranularite': ['annee', None], 'Scope': ['date', None]}
LowGranularite et Scope spatiaux : {'LowGranularite': ['departements', 'Ain'], 'Scope': ['departements', 'Ain']}

Attributs temporels : {}
Attributs spatiaux : {'Ain': ['alpes-maritimes', 'departements']}
LowGranularite et Sco

## 3. Enregistrer les données dans la classe Dataset

### 3.1 Créer un objet DS_Spatial_Scope